# the binary prediction problem using bio-signal----take smoker status data as an example

## the framework

abstract

keyword
(no more than 5)

1. introduction
(including background, essay content and main contribution)

2. related work
(a literiture review)

3. problem and solution
- 3.1 dataset
- 3.2 machine learning methods
- - 3.2.1 decision tree
- - 3.2.2 XGBoost
- - 3.2.3 random forest
- - 3.2.4 naive bayes
- - 3.2.5 KNN
- - 3.2.6 logistic regression
- - 3.2.7 SVM
- - 3.2.8 discriminant analysis
- - 3.2.9 MLP
- - 3.2.10 CNN
- 3.3 data pre-processing

4. experiment
- 4.1 descriptive analysis of dataset
- 4.2 performance evaluation and comparison
- - 4.2.1 evalusion criterior
- - 4.2.2 result

5. conclusion

6. futher discussion

reference

## the code

package

In [2]:
#import some nessisary package
import pandas as pd
import numpy as np
#visualisation
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import math

#######################################因无法下载相关包被我注释掉了
#描述性统计板块EDA
#from collections import Counter
#import pandas_profiling as pp
#######################################

# data preprocessing
from sklearn.preprocessing import StandardScaler

#import evaluation index such as confusion matrix, accuracy_score, f1_score, precision_score and recall_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# data splitting
from sklearn.model_selection import train_test_split

#import important machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier

导入/合并18个数据集

In [3]:
#import the csv into jupyter notebook
#the dataset
import pandas as pd

merged_df = pd.DataFrame()

# 循环读取并合并18个数据集
for i in range(501, 503):
    # 读取数据集
    df = pd.read_csv(f'/Users/zhengyawen/Desktop/zywhar70/har70plus/{i}.csv')
    
    # 第一个数据集直接赋值给merged_df
    if i == 501:
        merged_df = df
    else:
        # 根据共同列进行合并
        merged_df = pd.concat([merged_df,df],ignore_index=True)

# 输出合并后的数据集
print(merged_df)


处理缺失值

In [4]:
#data pre-processing
#check whether the dataset exist missing value
# 查看每列缺失值的数量
missing_values = merged_df.isnull().sum()
# 查看整个数据集缺失值的总数量
total_missing = merged_df.isnull().sum().sum()
# 打印缺失值情况
print(missing_values)
print(total_missing)

降噪

In [4]:
#求504人back传感器的合向量的模

#导入504df
df_504 = pd.read_csv(f'/Users/zhengyawen/Desktop/zywhar70/har70plus/504.csv')

# 计算欧氏距离的函数
def euclidean_distance(row):
    return np.sqrt(row['back_x']**2 + row['back_y']**2 + row['back_z']**2)

# 应用函数并将结果添加为新列
df_504['euclidean_distance'] = df_504.apply(euclidean_distance, axis=1)

print(df_504)

In [14]:
#对于每个label，我都画出对应的时间戳和欧氏距离的time series plot
# 循环遍历每个 label
for label in [1, 3, 4, 5, 6, 7, 8]:
    # 筛选出指定 label 的数据子集
    subset = df_504[df_504['label'] == label]
    
    # 绘制时间序列图
    plt.figure(figsize=(10, 6))
    plt.plot(subset['timestamp'], subset['euclidean_distance'], marker='o', linestyle='-')
    plt.title(f'Time Series Plot for Label {label}')
    plt.xlabel('Timestamp')
    plt.ylabel('Euclidean Distance')
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


重抽样技术（10-fold CV）

In [5]:
from sklearn.model_selection import KFold

# 假设你有一个特征矩阵X和对应的目标向量y
X = merged_df.drop(columns=['timestamp','label'])  # 删除目标列以获取特征矩阵
y = merged_df['label']  # 获取目标向量

# 初始化十折交叉验证对象
kf = KFold(n_splits=10, shuffle=True, random_state=42)

统计学分析（共线性检验）

In [6]:
# 使用corr()函数计算相关系数矩阵
correlation_matrix = merged_df.drop(columns=['label']).corr()

# 打印相关系数矩阵
print(correlation_matrix)

# 使用Seaborn的heatmap函数绘制热力图
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")

# 设置热力图的标题
plt.title('Correlation Heatmap')

# 显示图形
plt.show()

In [7]:
#使用VIF（方差膨胀因子）检验共线性
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 添加一个截距列，否则VIF求不出来，VIF是基于线性回归模型的
X_vif = merged_df.drop(columns=['timestamp','label'])
X_vif['Intercept'] = 1

# 初始化一个DataFrame来存储VIF结果
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns

# 计算每个变量的VIF
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

# 打印结果
print(vif_data)

    Variable        VIF
0     back_x   2.701746
1     back_y   1.206871
2     back_z   4.406978
3    thigh_x   2.049999
4    thigh_y   1.150039
5    thigh_z   2.122619
6  Intercept  23.288170


结果表明变量未存在明显的多重共线性（VIF<10），因此我们考虑使用多分类logistic模型

In [8]:
#统计学方法：使用多分类logistic进行分类
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# 初始化你要用的分类器，比如随机森林
classifier = LogisticRegression(multi_class='multinomial', solver='sag', max_iter=1000)

# 已经有了十折交叉验证对象
# 使用交叉验证进行模型评估
scores = cross_val_score(classifier, X, y, cv=kf)

# 输出十折交叉验证的准确率
print("Cross-validated scores:", scores)
print("Mean accuracy:", np.mean(scores))

Cross-validated scores: [0.8275305  0.82927348 0.82816818 0.82327934 0.82612762 0.83059134
 0.8271479  0.83126435 0.8273106  0.82879857]
Mean accuracy: 0.8279491874910626


机器学习算法（naive bayes）

In [ ]:
#NB


机器学习算法（decision tree）

In [16]:
#decision tree

机器学习算法（XGB）

In [ ]:
#XGB

机器学习算法（random forest）

In [ ]:
#RF

机器学习算法（KNN）

In [ ]:
#KNN

机器学习算法（SVM）

In [ ]:
#SVM

机器学习算法（QDA）

In [ ]:
#QDA

机器学习算法（LDA）

In [ ]:
#LDA

In [4]:
#data pre-processing
#check whether the dataset exist missing value
# 查看每列缺失值的数量
missing_values = df.isnull().sum()
# 查看整个数据集缺失值的总数量
total_missing = df.isnull().sum().sum()
# 打印缺失值情况
print(missing_values)
print(total_missing)


#check the dipulated value
#it's impossible to have dipulated value (i.e.the two same rows), so we didn't cost the time in finding them


#check the format of the data
#it seems that all the data are in a regular format, it's unnessisary to change the format


#standardize
#我只对['HDL', 'LDL','AST','ALT','Gtp']这五个变量进行标准化，因为他们最大值最小值的尺度差异太大
# 指定需要标准化的变量索引
columns_to_standardize = ['back_x','back_y','back_z','thigh_x','thigh_y','thigh_z']
# 创建标准化器
scaler = StandardScaler()
# 在训练集上拟合并进行标准化，仅对指定的列进行操作
X_train[columns_to_standardize] = scaler.fit_transform(X_train[columns_to_standardize])
# 在测试集上使用相同的均值和标准差进行标准化，仅对指定的列进行操作
X_test[columns_to_standardize] = scaler.fit_transform(X_test[columns_to_standardize])

In [6]:
#data visualization
#draw the line gragh in order to see the pattern
x = range(1,150759)
plt.figure(figsize=(20, 10), dpi=800)        #New canvas
plt.plot(x,df['back_x'],lw=1,label='the force in back in x axis')    #Set the line plot                      
plt.xlabel('time: in days during 2016/1/4 to 2022/12/30',size=20)
plt.ylabel('factors index points',size=20)
plt.legend(prop = {'size':14})
plt.grid()
plt.title("time series plot: factors index points",size=20)    #set the name of the graph
plt.show()

In [29]:
#descriptive statistic
description = X_train.describe()
print(description)

In [30]:
#some way to generate evaluation matrix
#confusion matrix, accuracy_score, f1_score, precision_score, recall_score
def get_metrics(y_test, y_predicted):
    accuracy = accuracy_score(y_test, y_predicted)
    precision = precision_score(y_test, y_predicted, average='weighted')
    recall = recall_score(y_test, y_predicted, average='weighted')
    f1 = f1_score(y_test, y_predicted, average='weighted')
    return accuracy, precision, recall, f1
def print_confusion(y_test,y_pred):
    print("Confusion matrix\n")
    print(pd.crosstab(pd.Series(y_test, name='Actual'), pd.Series(y_pred, name='Predicted')))
    accuracy, precision, recall, f1 = get_metrics(y_test, y_pred)
    print("accuracy = %.3f \nprecision = %.3f \nrecall = %.3f \nf1 = %.3f" % (accuracy, precision, recall, f1))

In [31]:
#KNN
print(X_train.shape)
print(Y_train.shape)
m1 = 'K-NeighborsClassifier'
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train, Y_train)
knn_predicted = knn.predict(X_test)
print_confusion(Y_test,knn_predicted)

In [32]:
#K为几的时候分类的准确度最高
# 定义 K 值的范围
k_values = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]

best_accuracy = 0
best_k = 0

for k in k_values:
    m1 = f'K-NeighborsClassifier (K={k})'
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, Y_train)
    knn_predicted = knn.predict(X_test)
    accuracy = accuracy_score(Y_test, knn_predicted)
    
    print(f"{m1} Accuracy: {accuracy:.4f}")


In [33]:
#logistic regression
m2 = 'Logistic Regression'
lr = LogisticRegression()
model = lr.fit(X_train, Y_train)
lr_predict = lr.predict(X_test)
print_confusion(Y_test,lr_predict)

iterations = lr.n_iter_[0]
print(f"Logistic Regression reached optimal solution in {iterations} iterations.")

In [34]:
#Decision tree
m3 = 'DecisionTreeClassifier'
dt = DecisionTreeClassifier(criterion = 'entropy',random_state=0,max_depth = 6)
dt.fit(X_train, Y_train)
dt_predicted = dt.predict(X_test)
print_confusion(Y_test,dt_predicted)

In [35]:
#XGBoost
m4 = 'Extreme Gradient Boost'
xgb = XGBClassifier(learning_rate=0.01, n_estimators=25, max_depth=15,gamma=0.6, subsample=0.52,colsample_bytree=0.6,seed=27, 
                    reg_lambda=2, booster='dart', colsample_bylevel=0.6, colsample_bynode=0.5)
xgb.fit(X_train, y_train)
xgb_predicted = xgb.predict(X_test)
print_confusion(y_test,xgb_predicted)

In [36]:
#Random Forest
m5 = 'Random Forest Classfier'
rf = RandomForestClassifier(n_estimators=20, random_state=12,max_depth=5)
rf.fit(X_train,Y_train)
rf_predicted = rf.predict(X_test)
print_confusion(Y_test,rf_predicted)

In [37]:
#Naive Bayes
m6 = 'Naive Bayes'
nb = GaussianNB()
nb.fit(X_train,Y_train)
nbpred = nb.predict(X_test)
print_confusion(Y_test,nbpred)

In [38]:
#SVM
m7 = 'Support Vector Classifier'
svc =  SVC(kernel='rbf', C=2)
svc.fit(X_train, Y_train)
svc_predicted = svc.predict(X_test)
print_confusion(Y_test,svc_predicted)

In [39]:
#linear descriminant analysis
m8 = 'Linear Discriminant Analysis'
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, Y_train)
lda_predicted = lda.predict(X_test)
print_confusion(Y_test,lda_predicted)

In [40]:
#quadratic discriminant analysis
m9 = 'quadratic discriminant analysis'
qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train, Y_train)
y_predicted = qda.predict(X_test)
print_confusion(Y_test,y_predicted)

In [41]:
#MLP
m10 = 'Multilayer Perceptron'
mlp = MLPClassifier(hidden_layer_sizes=(100, ), max_iter=500, random_state=42)
mlp.fit(X_train, Y_train)
y_predicted = mlp.predict(X_test)
print_confusion(Y_test,y_predicted)

In [129]:
#CNN
#我尝试了很多很多次，但是维度都不太对，我觉得这真的没办法怨天尤人，只能说自己的知识能力还是太浅薄了，只有更加努力地踏踏实实地
#去学习才可以解决这个问题，还不如稳住心态，把剩下的东西写好
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score
import numpy as np

# 假设你有训练数据 X_train, Y_train 和测试数据 X_test, Y_test
# 将数据转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test.to_numpy(), dtype=torch.float32)

# 定义 CNN 模型
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv1d(22, 32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

# 实例化模型、损失函数和优化器
model = SimpleCNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 将数据放入 DataLoader
train_dataset = TensorDataset(X_train_tensor.unsqueeze(1), Y_train_tensor.unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 训练模型
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs.permute(0, 2, 1))
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# 在测试集上进行预测
model.eval()
test_outputs = []
with torch.no_grad():
    for inputs, labels in zip(X_test_tensor.unsqueeze(1), Y_test_tensor.unsqueeze(1)):
        outputs = model(inputs)
        test_outputs.extend(outputs.cpu().numpy())

test_predictions = np.array(test_outputs) > 0.5
test_accuracy = accuracy_score(Y_test.numpy(), test_predictions)
print(f"Test Accuracy: {test_accuracy:.4f}")


In [136]:


imp_feature = pd.DataFrame({'Feature': ['age','height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation',
                                        'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries'], 'Importance': dt.feature_importances_})
plt.figure(figsize=(10,4))
plt.title("barplot Represent feature importance ")
plt.xlabel("importance ")
plt.ylabel("features")
plt.barh(imp_feature['Feature'],imp_feature['Importance'])
plt.show()

In [130]:
column_names = df.columns.tolist()
print(column_names)